In [3]:
# BAN6800 – BA-04: Data Understanding and Quality Profiling
# CGSL Predictive Maintenance Project

import pandas as pd
import numpy as np

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [5]:
# Load the UCI APS training dataset from GitHub
# The official UCI CSV contains 20 documentation lines before the header.

train_url = "https://raw.githubusercontent.com/dvfekorigha-create/BAN6800-cgsl-predictive-maintenance/main/data/raw/aps_failure_training_set.csv"

df_train = pd.read_csv(
    train_url,
    skiprows=20,
    na_values="na"
)

print("Training dataset loaded successfully.")
print("Shape:", df_train.shape)

Training dataset loaded successfully.
Shape: (60000, 171)


In [6]:
# Load the UCI APS test dataset from GitHub

test_url = "https://raw.githubusercontent.com/dvfekorigha-create/BAN6800-cgsl-predictive-maintenance/main/data/raw/aps_failure_test_set.csv"

df_test = pd.read_csv(
    test_url,
    skiprows=20,
    na_values="na"
)

print("Test dataset loaded successfully.")
print("Shape:", df_test.shape)

Test dataset loaded successfully.
Shape: (16000, 171)


In [7]:
# Verify dataset structure and target variable

print("TRAINING DATASET")
print("Rows and columns:", df_train.shape)

print("\nTEST DATASET")
print("Rows and columns:", df_test.shape)

print("\nTARGET VARIABLE")
print("Training target counts:")
print(df_train["class"].value_counts())

print("\nTraining target percentages:")
print((df_train["class"].value_counts(normalize=True) * 100).round(2))

print("\nTarget column in test dataset:", "class" in df_test.columns)

TRAINING DATASET
Rows and columns: (60000, 171)

TEST DATASET
Rows and columns: (16000, 171)

TARGET VARIABLE
Training target counts:
class
neg    59000
pos     1000
Name: count, dtype: int64

Training target percentages:
class
neg    98.33
pos     1.67
Name: proportion, dtype: float64

Target column in test dataset: True


In [8]:
# Profile missing values in the training dataset

missing_count = df_train.isna().sum()
missing_percent = (df_train.isna().mean() * 100).round(2)

missing_profile = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing Percent": missing_percent
})

# Show only columns containing missing values
missing_profile = missing_profile[missing_profile["Missing Count"] > 0] \
    .sort_values("Missing Percent", ascending=False)

print("Number of columns containing missing values:", len(missing_profile))
print("\nTop 20 columns by missing percentage:")
display(missing_profile.head(20))

Number of columns containing missing values: 169

Top 20 columns by missing percentage:


,Missing Count,Missing Percent
br_000,49264,82.11
bq_000,48722,81.20
bp_000,47740,79.57
bo_000,46333,77.22
cr_000,46329,77.22
ab_000,46329,77.22
bn_000,44009,73.35
bm_000,39549,65.92
bl_000,27277,45.46
bk_000,23034,38.39


In [9]:
# Check for duplicate records

duplicate_count = df_train.duplicated().sum()
duplicate_percent = (duplicate_count / len(df_train)) * 100

print("Duplicate rows:", duplicate_count)
print("Duplicate percentage:", round(duplicate_percent, 2), "%")

Duplicate rows: 0
Duplicate percentage: 0.0 %


In [10]:
# Check data types and constant features

print("DATA TYPES")
print(df_train.dtypes.value_counts())

# Count unique values in each column
unique_counts = df_train.nunique(dropna=False)

constant_features = unique_counts[unique_counts == 1]

print("\nNumber of constant features:", len(constant_features))

if len(constant_features) > 0:
    print("\nConstant features:")
    print(constant_features)
else:
    print("No constant features found.")

DATA TYPES
float64    169
object       1
int64        1
Name: count, dtype: int64

Number of constant features: 0
No constant features found.


In [11]:
# Validate target variable values

print("Unique target values:")
print(df_train["class"].unique())

print("\nNumber of unique target values:")
print(df_train["class"].nunique())

print("\nTarget value counts including missing values:")
print(df_train["class"].value_counts(dropna=False))

Unique target values:
['neg' 'pos']

Number of unique target values:
2

Target value counts including missing values:
class
neg    59000
pos     1000
Name: count, dtype: int64


In [12]:
# Check numeric feature ranges and identify columns with no observed numeric values

numeric_df = df_train.drop(columns=["class"])

summary = numeric_df.describe().T

range_check = summary[["min", "max"]].copy()

print("Number of numeric features:", numeric_df.shape[1])

print("\nFeatures with minimum values below zero:")
negative_min = range_check[range_check["min"] < 0]
print(negative_min if not negative_min.empty else "None found")

print("\nFeatures with maximum values above 1,000,000:")
large_max = range_check[range_check["max"] > 1_000_000]
print(large_max if not large_max.empty else "None found")

Number of numeric features: 170

Features with minimum values below zero:
None found

Features with maximum values above 1,000,000:
        min           max
aa_000  0.0  2.746564e+06
ac_000  0.0  2.130707e+09
ad_000  0.0  8.584298e+09
ag_000  0.0  3.376892e+06
ag_001  0.0  4.109372e+06
...     ...           ...
ee_005  0.0  5.743524e+07
ee_006  0.0  3.160781e+07
ee_007  0.0  1.195801e+08
ee_008  0.0  1.926740e+07
ee_009  0.0  3.810078e+06

[134 rows x 2 columns]


In [13]:
# Summarize extreme missingness

missing_summary = df_train.isna().mean() * 100

over_50 = (missing_summary > 50).sum()
over_80 = (missing_summary > 80).sum()

print("Features with more than 50% missing values:", over_50)
print("Features with more than 80% missing values:", over_80)

print("\nHighest missingness percentage:")
print(missing_summary.sort_values(ascending=False).head(10).round(2))

Features with more than 50% missing values: 8
Features with more than 80% missing values: 2

Highest missingness percentage:
br_000    82.11
bq_000    81.20
bp_000    79.57
bo_000    77.22
ab_000    77.22
cr_000    77.22
bn_000    73.35
bm_000    65.92
bl_000    45.46
bk_000    38.39
dtype: float64


## BA-04 Data Quality and Class-Balance Findings

Initial profiling of the UCI APS Failure at Scania Trucks training dataset identified the following findings:

- The training dataset contains 60,000 observations and 171 columns.
- There are 170 numeric operational features and one categorical target variable, `class`.
- The target contains two valid classes: `neg` and `pos`.
- The training target is highly imbalanced: 59,000 `neg` observations (98.33%) and 1,000 `pos` observations (1.67%).
- Missing values are widespread: 169 of the 171 columns contain missing values.
- Eight features have more than 50% missing values, while two features have more than 80% missing values.
- No duplicate rows were identified.
- No constant features were identified.
- No negative minimum values were observed in the numeric-feature screening.
- 134 numeric features have observed maximum values above 1,000,000. Because the operational feature meanings are anonymized, these values are treated as high-range observations requiring further investigation rather than automatically classified as errors.
- The target variable contains no missing observations.

### Implications for the Project

The profiling results indicate that the next stage requires a documented missing-value preprocessing strategy, feature-scale assessment and careful handling of the strong target-class imbalance. Accuracy alone will not be sufficient for model evaluation; precision, recall, F1-score, ROC-AUC and cost-sensitive evaluation will be emphasized.

The findings from this profiling stage will inform BA-05 (missing-value preprocessing), BA-06 (exploratory data analysis), and subsequent modelling activities.

In [14]:
# Create a reproducible BA-04 profiling report

profiling_report = pd.DataFrame({
    "Metric": [
        "Training rows",
        "Total columns",
        "Numeric features",
        "Target classes",
        "Positive cases",
        "Positive class percentage",
        "Negative cases",
        "Negative class percentage",
        "Columns with missing values",
        "Features >50% missing",
        "Features >80% missing",
        "Duplicate rows",
        "Constant features",
        "Numeric features with max > 1,000,000"
    ],
    "Value": [
        len(df_train),
        df_train.shape[1],
        df_train.drop(columns=["class"]).shape[1],
        df_train["class"].nunique(),
        (df_train["class"] == "pos").sum(),
        round((df_train["class"] == "pos").mean() * 100, 2),
        (df_train["class"] == "neg").sum(),
        round((df_train["class"] == "neg").mean() * 100, 2),
        df_train.isna().any().sum(),
        (df_train.isna().mean() > 0.50).sum(),
        (df_train.isna().mean() > 0.80).sum(),
        df_train.duplicated().sum(),
        (df_train.nunique(dropna=False) == 1).sum(),
        (df_train.drop(columns=["class"]).max(numeric_only=True) > 1_000_000).sum()
    ]
})

display(profiling_report)

# Save report as CSV
profiling_report.to_csv("BA-04_data_quality_summary.csv", index=False)

# Save detailed feature-level missingness profile
missing_profile.to_csv("BA-04_missingness_profile.csv")

print("\nFiles created successfully:")
print("- BA-04_data_quality_summary.csv")
print("- BA-04_missingness_profile.csv")

,Metric,Value
0,Training rows,60000.00
1,Total columns,171.00
2,Numeric features,170.00
3,Target classes,2.00
4,Positive cases,1000.00
5,Positive class percentage,1.67
6,Negative cases,59000.00
7,Negative class percentage,98.33
8,Columns with missing values,169.00
9,Features >50% missing,8.00



Files created successfully:
- BA-04_data_quality_summary.csv
- BA-04_missingness_profile.csv


In [15]:
from google.colab import files

files.download("BA-04_data_quality_summary.csv")
files.download("BA-04_missingness_profile.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>